## Setup

### Load Modules

In [3]:
%load_ext autoreload
%autoreload 2

#General Import
import os
import numpy as np
import matplotlib.pyplot as plt
import pickle
from os.path import join
from sklearn.model_selection import KFold
from matplotlib import colormaps as cmaps
from mne.filter import filter_data, resample
import scipy.stats as stats
import pandas as pd
from itertools import product
import xarray as xr
from scipy.signal import coherence, welch
from matplotlib import cm
from matplotlib.ticker import LinearLocator
import re
import colorcet as cc
from wordfreq import word_frequency
from pingouin import bayesfactor_ttest

#ML Import
from sklearn.decomposition import PCA, FastICA, SparsePCA, FactorAnalysis, NMF
from sklearn.preprocessing import StandardScaler, scale
from sklearn.metrics import silhouette_score
from dtaidistance.preprocessing import differencing
import dtaidistance.clustering.kmeans as dtwkmeans
import dtaidistance as dta
import jPCA
import scipy.signal as signal
from statsmodels.tsa.stattools import grangercausalitytests
import tslearn
from statsmodels.stats.multitest import multipletests

#Electrophysiology Import
from spyeeg.models.TRF import TRFEstimator
from spyeeg.models.ERP import ERP_class
from spyeeg.utils import lag_matrix
import mne
import frites
from frites.simulations import sim_multi_suj_ephy
from frites.dataset import DatasetEphy
from frites.workflow import WfConnComod
from frites import set_mpl_style
import frites.conn as conn
from scipy.signal import welch
import spectral_connectivity 

#Graph Import
import networkx as nx
from matplotlib_venn import venn3, venn3_circles, venn2
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
import colorcet as cc
import seaborn as sns

#Performance Import
import time
import psutil

#Local Import
from stats_utils import cliffs_delta, cohen_d, max_cluster, select_clusters, filter_binary
from nice_utils import estimate_loop_time, decorator_loop
from preprocessing_utils import mono_to_bipolar, select_channels, available_regions, delete_channels, adj_scale, ica_shaft, get_bipolar_atlas
from signal_utils import sparse_resample, lag_finder
from viz_utils import create_matshow_gif, create_collection_gif, _arrow3D


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load Features

In [ ]:
# Acoustic Regressors
fs = 100
new_path = 'C:/Users/D-CAP/Documents/GitHub/witching-star/regressors/selected_regs.pkl'
new_data = pickle.load(open(new_path, 'rb'))
data_fs = new_data['fs']
new_regressors = new_data['regs']
new_names = new_data['regs_name']
ratio = fs/data_fs
new_duration = int(new_regressors.shape[0] * ratio) + 1

new_resamp = []
for i in range(new_regressors.shape[1]):
    name = new_names[i]
    if name in ['Intensity', 'Envelope Oganian', 'Envelope Derivative TF', 'F0 Loudness', 'SpectralFlux Filtered', 'SpectralFlux not_filtered']:
        new_reg = mne.filter.resample(new_regressors[:,i], up=100, down=data_fs)[:new_duration]
    elif name in ['peakEnv_tf', 'Syllabe Onset', 'p-syl', 'Phono']:
        new_reg = new_regressors[:,i] - np.min(new_regressors[:,i])
        new_reg = sparse_resample(new_reg, new_fs = fs, current_fs = data_fs)[:new_duration]
    else:
        print('wut')
    new_resamp.append(new_reg)
new_resamp = np.asarray(new_resamp).T
regressors = new_resamp
regressors_name = new_data['regs_name']


In [ ]:
# Renyi2 Array

path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/renyi_array2.pickle"
renyi_data = pickle.load(open(path_renyi, 'rb'))
data_fs = renyi_data['fs']
X = np.roll(renyi_data['X'],4, axis=0)
onsets = np.where(X[:,5] >0)[0]
reg_renyi = np.zeros([regressors.shape[0], X.shape[1]])

for onset in onsets:
    new_onset = int(onset/data_fs*fs)
    for renyi_index in range(X.shape[1]):
        reg_renyi[new_onset, renyi_index] = X[onset,renyi_index]

regressors = np.hstack([regressors, reg_renyi])
regressors_name = regressors_name + renyi_data['names']
renyi_values_wrd = [float(renyi_name.split('renyi ')[1]) for renyi_name in renyi_data['names'][:20]]

### Load Neural Data

In [ ]:
# Load Broadband

data_subject = dict()
channels_subject = dict()
locations_subject = dict()
path_data = "D:/DataSEEG_Sorciere/BIDS/data_mne_fif"
for index_subject in range(1,40):
    try:
        name_subject = 'sub-{:03}'.format(index_subject)
        name_file = name_subject + '_task-iSpeech_speech-epo.fif'
        path_file = os.path.join(path_data, name_subject,'preprocessed','epochs',name_file)
        if os.path.isfile(path_file):
            mne_data = mne.read_epochs(path_file, verbose = False)
        elif os.path.isfile(os.path.join(path_data, name_subject,'preprocessed','monopolar',name_file)):
            path_file = os.path.join(path_data, name_subject,'preprocessed','monopolar',name_file)
            mne_data = mne.read_epochs(path_file, verbose = False)
        else:
            path_file = os.path.join(path_data, name_subject,'preprocessed','epochs','monopolar',name_file)
            mne_data = mne.read_epochs(path_file, verbose = False)
        mne_data_resample = mne_data.resample(fs, npad = 'auto', verbose = False)
        mne_data_resample.filter(0.3, 49, verbose = False)
        channels = mne_data_resample.ch_names
        montage = mne_data_resample.get_montage()
        ch_names = [n for n,_ in montage.get_positions()['ch_pos'].items()] 
        loc = (1e3 * np.stack([coord for _,coord in montage.get_positions()['ch_pos'].items()])).T  # store locations
        
        data_subject[index_subject] = mne_data_resample.get_data(copy = False)[0].T[:regressors.shape[0],:]
        channels_subject[index_subject] = channels
        locations_subject[index_subject] = loc
        print('Subject', index_subject, 'loaded')
    except:
        print('Error in subject', index_subject)

data_bipolar, channels_bipolar, locations_bipolar = mono_to_bipolar(data_subject, channels_subject, locations_subject)

In [ ]:
atlas_subject = dict()
path_data = "D:/DataSEEG_Sorciere/BIDS/data_mne_fif"
path_atlas = "D:/DataSEEG_Sorciere/BIDS/freesurfer"
for index_subject in range(1,40):
    try:
        name_subject = 'sub-{:03}'.format(index_subject)
        name_bipolar_atlas = 'elecbipolar2atlas.mat'
        name_monopolar_atlas = 'elec2atlas.mat'
        path_bipolar_atlas = os.path.join(path_atlas, name_subject,name_bipolar_atlas)
        path_monopolar_atlas = os.path.join(path_atlas, name_subject,name_monopolar_atlas)
        bipolar_atlas = get_bipolar_atlas(path_bipolar_atlas, atlas = 'Desikan_Killiany') #Destrieux
        atlas_subject[index_subject] = bipolar_atlas
    except:
        print('Atlas error in subject', index_subject)

subjects =  list(atlas_subject.keys())
new_atlas = dict()
for subject_index in atlas_subject:
    new_atlas[subject_index] = dict()
    for channels_bipolar in atlas_subject[subject_index]:
        new_name = channels_bipolar.split('-')[0] + '||' + channels_bipolar.split('-')[1]
        new_atlas[subject_index][new_name] = atlas_subject[subject_index][channels_bipolar][0]

## Mutual Information Extraction

### Load Files

In [ ]:
channel_selection = ['H','T']
#channel_selection = ['H','T','OP', 'OT', 'OR', 'OC', 'TP', 'TB', 'GPH']
channel_selection = []
exclude = False

reg_label_dict = {233: 'Dispersion', 234: 'renyi WRD 0.016',235: 'renyi WRD 0.0263',236: 'renyi WRD 0.0428',237: 'renyi WRD 0.0695',238: 'renyi WRD 0.1128',239: 'renyi WRD 0.1832',
                  240: 'renyi WRD 0.2976',241: 'renyi WRD 0.4832',242: 'renyi WRD 0.7847',243: 'renyi WRD 1.2742',244: 'Shannon',245: 'renyi WRD 3.3598',246: 'renyi WRD 5.4555',
                  247: 'renyi WRD 8.8586',248: 'renyi WRD 14.384',249: 'renyi WRD 23.357',250: 'renyi WRD 37.926',251: 'renyi WRD 61.584',252: 'renyi WRD 100.0',
                  253: 'Strength',254: 'Strength 2',255: 'Min15Entro',256: 'Max2Entro',
                  257: 'Shannon Entropy',258: 'Surprisal',259: 'Lexical Surprise'}

montage = 'bipo' #bipo, mono, ica, 'Hfa_bipo'
white_flag, grey_flag = 1,1

pre_clustering = False
clustering = 'Venn'
montage_choice = 'bipo'
regressors_clusters = [873]

regressors_clustering_str = '_'.join(np.asarray(regressors_clusters).astype('str'))
n_perm = 1000

subjects_indices = np.arange(len(data_bipolar))
regressors_list = list(np.arange(233,253)) + [258]


renyi_values = renyi_values_wrd

apply_baseline = True
baseline_str = (not apply_baseline) * 'no_baseline'
mi_group = dict()
skip_symmetry = True
cat = '' #'cat_' or ''
frequency_band = '' #low_gamma-band_ or beta-band_
channel_selection_join = ''.join(channel_selection)
for subject_index in range(len(data_subject)):
#for subject_index in range(25):
    subject_id = list(data_subject.keys())[subject_index]
    mi_group[subject_id] = dict()
    eeg = data_subject[subject_id]
    channels = channels_subject[subject_id]
    locations = locations_subject[subject_id]
    eeg_selection, channels_selection, locations_selection = select_channels(eeg, channels, locations, channel_select = channel_selection, exclude = exclude)
    if skip_symmetry or ("H1" in channels and "H'1" in channels):
        for regressor_index in regressors_list:
            #filename = 'MIStat/MIStat_' + str(n_perm) + '_' + montage + '_' + channel_selection_join + '_reg' + str(regressor_index) + '_sub' + str(subject_id) + '.pickle'
            filename = 'D:/Data_MI/MIStat_' + str(n_perm) + '_' + montage + '_' + channel_selection_join + '_reg' + str(regressor_index) + '_sub' + str(subject_id) + '.pickle'
            #filename = 'D:/Data_MI/MIStat2_' + baseline_str + str(n_perm) + '_' + montage + '_' + channel_selection_join + '_reg' + str(regressor_index) + '_sub' + str(subject_id) + '.pickle'
            filename = 'D:/Data_MI/MIStat3_' + baseline_str + str(n_perm) + '_' + montage + '_' + channel_selection_join + '_reg' + str(regressor_index) + '_sub' + str(subject_id) + '.pickle'
            if pre_clustering:
                target_name = clustering + '_' + montage_choice + '_' +  str(subject_id) + '_' + regressors_clustering_str 
                filename = 'D:/Data_MI/MIStat3_' + baseline_str + str(n_perm) + '_' + montage + '_' + target_name + '_reg' + str(regressor_index) + '_sub' + str(subject_id) + '.pickle'
            with open(filename, 'rb') as file:
                mi = pickle.load(file)
            if 'permut' in mi and manual_stat:
                mi_data = np.asarray(mi['mutual']).T
                mi_permut = np.asarray(mi['permut'])[:,:,0,:]
                mi_pval = np.ones(mi['pval'].data.shape)
                stat_mask, perm_dist, no_perm_dist = select_clusters(mi_data, mi_permut, thres = 0.01, pval = 0.05)
                mi_pval[:,stat_mask] = 0
                mi['pval'].data = mi_pval
            if 'permut' in mi:
                mi_group[subject_id][regressor_index] = {'data' : mi['mutual'], 'pval' : mi['pval']}



for subject_id in mi_group:
    for regressor_index in mi_group[subject_id]:
        indices = []
        rois = mi_group[subject_id][regressor_index]['data'].roi.data
        for name_index, name in enumerate(rois):
            if name in new_atlas[subject_id]:
                if 'no_label_found' in new_atlas[subject_id][name] or 'nknown' in new_atlas[subject_id][name]:
                    True
                elif not white_flag and ('White-Matter' in new_atlas[subject_id][name]):
                    True
                elif white_flag and ('White-Matter' in new_atlas[subject_id][name]):
                    indices.append(name_index)
                elif grey_flag and not ('White-Matter' in new_atlas[subject_id][name]):
                    indices.append(name_index)
                elif not grey_flag and not ('White-Matter' in new_atlas[subject_id][name]):
                    True
                else:
                    print('jambon')
            else:
                indices.append(name_index)
        
        mi_group[subject_id][regressor_index] = {'data' : mi_group[subject_id][regressor_index]['data'][:,indices], 'pval' : mi_group[subject_id][regressor_index]['pval'][:,indices]}


### Helper and Color Dictionary

In [ ]:
timecourse_array = mi['mutual'].times.data/100 - 1
qualitative_map = cc.cm.rainbow #cm.tab10
qualitative_map2 = cc.cm.colorwheel #cm.tab10
qualitative_map3 = cc.cm.isolum #cm.tab10
continuous_map = cm.viridis
continuous_map2 = cm.jet
#continuous_map = cc.cm.CET_R3

ref_map = cc.cm.dimgray
color_shannon = [0.85,0.55,0.2]
color_surprisal = [0.4,0,0.4]


In [ ]:
reg_color = []
reg_color_dict = dict()
reg_label = []
for reg_index, reg in enumerate(regressors_list):
    reg_label.append(reg_label_dict[reg])
    color_value = int(reg_index/len(regressors_list)*200) #200
    reg_color.append(continuous_map(color_value))
    reg_color_dict[reg] = reg_color[reg_index]

reg_color2 = []
reg_color2_dict = dict()
for reg_index, reg in enumerate(regressors_list):
    color_value = int(reg_index/len(regressors_list)*200) #200
    reg_color2.append(continuous_map2(color_value))
    reg_color2_dict[reg] = reg_color[reg_index]

In [ ]:
cluster_color = []
cluster_color_dict = dict()
cluster_reg_color = []
cluster_reg_color_dict = dict()

n_c = 10
base_colors_1 = np.vstack([qualitative_map(np.asarray([0,1,0.25,0.75,0.5])),qualitative_map2(np.asarray([0,0.2,0.4,0.6,0.8])),qualitative_map3(np.asarray([0,0.2,0.4,0.6,0.8]))])
base_colors_2 = ref_map(np.ones(n_c)*0.8)


for cluster_index, color in enumerate(base_colors_1):
    cluster_color.append(color)
    cluster_color_dict[cluster_index] = color


def create_category_colormap(base_color, n_steps=256):
    return mcolors.LinearSegmentedColormap.from_list(
        f"modulated_{base_color}",
        [mcolors.to_rgba(base_color, alpha=0.2), mcolors.to_rgba(base_color, alpha=1.0)],
        N=n_steps,
    )

def create_mix_colormap(base_color_1, base_color_2, n_steps=256):
    return mcolors.LinearSegmentedColormap.from_list(
        f"modulated_{base_color_1}",
        [mcolors.to_rgba(base_color_1, alpha=0.8), mcolors.to_rgba(base_color_2, alpha=0.95)],
        N=n_steps,
    )

category_colormaps = {i: create_category_colormap(base_colors_1[i]) for i in range(n_c)}
category_colormaps = {i: create_mix_colormap(base_colors_1[i],base_colors_2[i]) for i in range(n_c)}

for cluster_index in range(n_c):
    cluster_reg_color_dict[cluster_index] = dict()
    cluster_reg_color.append([])
    for reg_index, reg in enumerate(regressors_list):
        color_value = int(reg_index/len(regressors_list)*256)
        cluster_reg_color[cluster_index].append(category_colormaps[cluster_index](color_value)) #coolwarm
        cluster_reg_color_dict[cluster_index][reg] = cluster_reg_color[cluster_index][reg_index]

cluster_reg_color[1] = cluster_reg_color[1][::-1]

In [ ]:
reg_label = []
for reg in regressors_list:
    reg_label.append(reg_label_dict[reg])
reg0 = regressors_list[0]

### Select Relevant Channels

In [ ]:
p_thres = 1/(n_perm - 1)
p_thres = 0.05
ignore_reg = [258]
#ignore_reg = []
#ignore_reg = list(np.arange(234,252)) + [258]

for subject_index, subject_id in enumerate(mi_group):
    for regressor_index, regressor_id in enumerate(mi_group[subject_id]):
        pvalues = mi_group[subject_id][regressor_id]['pval'].data.min(0)
        names = mi_group[subject_id][regressor_id]['pval'].roi.data
        times = mi_group[subject_id][regressor_id]['pval'].times.data
        channel_selection = {'pval': pvalues, 'names' : names, 'times': times}
        filename = 'MI_select/' + str(subject_id) + '_' + str(regressor_id) + '.pickle'
        with open(filename, 'wb') as file:
            pickle.dump(channel_selection,file)

In [ ]:
mi_thres = dict()
for subject_index, subject_id in enumerate(mi_group):
    mi_thres[subject_id] = dict()
    for regressor_index, regressor_id in enumerate(mi_group[subject_id]):
        select = mi_group[subject_id][regressor_id]['pval'].data.min(0) < p_thres
        names = mi_group[subject_id][regressor_id]['pval'].roi.data
        data = mi_group[subject_id][regressor_id]['data'].data
        data_select = data[:, select]
        names_select = names[select]
        mi_thres[subject_id][regressor_id] = {'data': data_select, 'names': names_select}

In [ ]:
mi_union_thres = dict()
for subject_index, subject_id in enumerate(mi_group):
    mi_union_thres[subject_id] = dict()
    select = np.zeros(mi_group[subject_id][regressor_id]['pval'].data.shape[1])
    for regressor_index, regressor_id in enumerate(mi_group[subject_id]):
        if not regressor_id in ignore_reg:
            select += (mi_group[subject_id][regressor_id]['pval'].data.min(0) < p_thres)
    select = list(np.clip(select,0,1).astype('bool'))
    for regressor_index, regressor_id in enumerate(mi_group[subject_id]):
        names = mi_group[subject_id][regressor_id]['pval'].roi.data
        data = mi_group[subject_id][regressor_id]['data'].data
        data_select = data[:, select]
        names_select = names[select]
        mi_union_thres[subject_id][regressor_id] = {'data': data_select, 'names': names_select}

In [ ]:
mi_inter_thres = dict()
for subject_index, subject_id in enumerate(mi_group):
    mi_inter_thres[subject_id] = dict()
    select = np.zeros(mi_group[subject_id][regressor_id]['pval'].data.shape[1])
    for regressor_index, regressor_id in enumerate(mi_group[subject_id]):
        if not regressor_id in ignore_reg:
            select += (mi_group[subject_id][regressor_id]['pval'].data.min(0) < p_thres)
    select = list(np.clip(select - len(mi_group[subject_id]) + len(ignore_reg) + 1,0,1).astype('bool'))
    for regressor_index, regressor_id in enumerate(mi_group[subject_id]):
        names = mi_group[subject_id][regressor_id]['pval'].roi.data
        data = mi_group[subject_id][regressor_id]['data'].data
        data_select = data[:, select]
        names_select = names[select]
        mi_inter_thres[subject_id][regressor_id] = {'data': data_select, 'names': names_select}

In [ ]:
mi_unique_thres = dict()
for subject_index, subject_id in enumerate(mi_group):
    mi_unique_thres[subject_id] = dict()
    for regressor_index, regressor_id in enumerate(mi_group[subject_id]):
        if not regressor_id in ignore_reg:
            names_delete = []
            names = mi_thres[subject_id][regressor_id]['names']
            data = mi_thres[subject_id][regressor_id]['data']
            for regressor_ref, regressor_ref_id in enumerate(mi_group[subject_id]):
                if regressor_ref_id != regressor_id and not regressor_ref_id in ignore_reg:
                    ref_names = mi_thres[subject_id][regressor_ref_id]['names']
                    for name in names:
                        if name in ref_names:
                            names_delete.append(name)
            names_delete = set(names_delete)
            data_select = []
            names_select = []
            for i in range(data.shape[1]):
                name = names[i]
                data_channel = data[:,i]
                if not name in names_delete:
                    data_select.append(data_channel)
                    names_select.append(name)
            data_select = np.asarray(data_select).T
            names_select = np.asarray(names_select)
            mi_unique_thres[subject_id][regressor_id] = {'data': data_select, 'names': names_select}

In [ ]:
all_names = dict()
venn_regressors = list(mi_thres[subject_id].keys())
for regressor_index, regressor_id in enumerate(venn_regressors):
    all_names[regressor_id] = []
for subject_index, subject_id in enumerate(mi_thres):
    for regressor_index, regressor_id in enumerate(venn_regressors):
        names = mi_thres[subject_id][regressor_id]['names']
        all_names[regressor_id] += [str(subject_id) + '_' + name for name in names]

set_list = []
set_names = []

%matplotlib qt
if len(all_names)>3:
    print('too many regressors')
elif len(all_names) == 1:
    print('The Venn diagram is a circle')
else:
    plt.figure()
    for regressor_index, regressor_id in enumerate(all_names):
        set_list.append(set(all_names[regressor_id]))
        set_names.append(reg_label_dict[regressor_id])
    if len(all_names) == 3:
        venn = venn3(set_list, set_labels=set_names)
    elif len(all_names) == 2:
        venn = venn2(set_list, set_labels=set_names)

## Grand Average

### Main Figure

In [ ]:
%matplotlib qt


timelock = 'onset'
fig = plt.figure(figsize=(10, 4.5))
gs = gridspec.GridSpec(1, 2, width_ratios=[2, 1.5])  # Custom ratios
axes = [fig.add_subplot(gs[0]), fig.add_subplot(gs[1])]
wrd_distrib = np.load('wrd_distrib.npy')
kernel = stats.gaussian_kde(wrd_distrib)
kernel_pdf = kernel.pdf(timecourse_array)
clip = False
norm = False
data_dict = mi_union_thres
max_chan = []
if clip:
    a = 0
    b = np.inf
else:
    a = -np.inf
    b = np.inf

avg_time = dict()
n_chan = dict()
max_chan = dict()
argmax_chan = dict()
for regressor_id in data_dict[list(data_dict.keys())[0]]:
    avg_time[regressor_id] = []
    n_chan[regressor_id] = 0
    max_chan[regressor_id] = []
    argmax_chan[regressor_id] = []
for subject_index, subject_id in enumerate(data_dict.keys()):
    for regressor_index, regressor in enumerate(data_dict[subject_id].keys()):
        if len(data_dict[subject_id][regressor]['names']) > 0:
            avg_time[regressor].append(np.mean(np.clip(data_dict[subject_id][regressor]['data'].data,a,b),axis=1))
            n_chan[regressor] += len(data_dict[subject_id][regressor]['names'])
            max_chan[regressor] += list(np.asarray(data_dict[subject_id][regressor]['data'].data).max(0))
            argmax_chan[regressor] += list(np.asarray(data_dict[subject_id][regressor]['data'].data).argmax(0))
            
ax = axes[0]
for reg_index, reg in enumerate(avg_time):
    info_time = scale(np.asarray(avg_time[reg]), with_mean=norm, with_std=norm,axis=1)
    if reg_index >= len(renyi_values):
        #ax.plot(timecourse_array,np.mean(info_time,axis=0), color = color_surprisal,linestyle = '--', linewidth = 3, label = 'Surprisal', alpha = 0)#, label = round(float(reg_label[reg_index][10:14]),2))
        ax.fill_between(timecourse_array,
                            np.mean(info_time,axis=0) + np.std(info_time,axis=0)/np.sqrt(info_time.shape[0]), 
                            np.mean(info_time,axis=0) - np.std(info_time,axis=0)/np.sqrt(info_time.shape[0]), 
                            color = color_surprisal, alpha = 0)
    else:
        ax.plot(timecourse_array,np.mean(info_time,axis=0), color = reg_color[reg_index])#, label = round(float(reg_label[reg_index][10:14]),2))
        ax.fill_between(timecourse_array,
                            np.mean(info_time,axis=0) + np.std(info_time,axis=0)/np.sqrt(info_time.shape[0]), 
                            np.mean(info_time,axis=0) - np.std(info_time,axis=0)/np.sqrt(info_time.shape[0]), 
                            color = reg_color[reg_index], alpha = 0.3)
if 1 in renyi_values:
    index_shannon = np.argmin(np.abs(np.asarray(renyi_values) - 1))
    shannon_time = scale(avg_time[list(avg_time.keys())[index_shannon]], with_mean=norm, with_std=norm,axis=1)
ax.plot(timecourse_array,np.mean(shannon_time,axis=0), color = color_shannon, label = 'Shannon Entropy', linestyle = '--', linewidth = 3, alpha = 1)
ax.plot(0,0,color = cm.viridis(125), label = 'Renyi Entropies', linewidth = 3,alpha = 0.8)

#Nomenclature
ax.set_xlabel('Time (s)', size = 18)
ax.set_ylabel('Mutual information (bits)', size = 18)
ax.spines[['right', 'top']].set_visible(False)
ax.spines[['bottom', 'left']].set_linewidth(2)
ax.tick_params(width=3, labelsize = 16)
ax.legend(loc='upper right',  prop={'size': 16}, fancybox=True, framealpha=0, bbox_to_anchor = (1.1,1))
#fig.suptitle('Mutual information for different Renyi parameters', size = 18)

#Xaxis
ax.set_xticks([-0.5,-0.25,0,0.25,0.5,0.75,1])
ax.set_xlim(-0.5,1.2)

#Yaxis
ax.set_yticks([0,0.005,0.010])
if timelock == 'onset':
    ax.plot([0,0],[0,0.0046], color = 'k', linestyle = '--')
    ax.text(-0.25,0.0049,'word\nonset', color = 'k', size = 18)
    for percentile in [95]:
        ax.plot([np.percentile(wrd_distrib,percentile),np.percentile(wrd_distrib,percentile)],[0,0.0046], color = 'k', linestyle = '--')
        ax.text(np.percentile(wrd_distrib,percentile)-0.05,0.0049,str(percentile) + '%\noffset', color = 'k', size = 18)
ax.set_ylim(0,0.008)
ax.set_facecolor((0.941, 0.969, 1.0))
fig.patch.set_facecolor((0.941, 0.969, 1.0))

#Kernel Handling
#ax.plot(timecourse_array,kernel_pdf/max(kernel_pdf)/100 + 0.014, color = 'k')

ax = axes[1]
reg_avg = np.zeros(len(regressors_list))
reg_std = np.zeros(len(regressors_list))
reg_max = np.zeros(len(regressors_list))

for regressor_index, regressor_id in enumerate(regressors_list):
    reg_avg[regressor_index] = np.mean(np.asarray(avg_time[regressor_id])[:,50:150])
    #reg_max[regressor_index] = np.max(np.asarray(avg_time[regressor_id]),axis=1).mean()
    reg_max[regressor_index] = np.max(np.asarray(avg_time[regressor_id])[:,120:130],axis=1).mean() #120:130
    reg_std[regressor_index] = np.std(np.max(np.asarray(avg_time[regressor_id]),axis=1)) / np.sqrt(len(avg_time[regressor_id]))

for renyi_val, avg_mi, std_mi, max_mi, renyi_id in zip(renyi_values, reg_avg, reg_std, reg_max, regressors_list):
    if renyi_val == 1:
        ax.scatter(renyi_val, max_mi, s= 200,color = color_shannon, zorder = 6, edgecolor = 'k', linewidth = 1.5)
        ax.errorbar(renyi_val, max_mi, yerr = std_mi, color = 'k', zorder = 5, linewidth = 1.5)
    else:
        ax.scatter(renyi_val, max_mi, s= 200,color = reg_color_dict[renyi_id], edgecolor = 'k',zorder = 2, linewidth = 1.5)
        ax.errorbar(renyi_val, max_mi, yerr = std_mi, color = 'k',zorder = 1, linewidth = 1.5)

ax.text(0.02,0.003,'    ' + str(np.sum([mi_union_thres[i][reg0]['data'].shape[1] for i in mi_union_thres])) + '\nchannels', size = 18, color = cm.viridis(100))
ax.set_xscale('log')
ax.set_xlabel('Renyi Order (α)', size = 18)
ax.set_ylabel('Peak of MI (bits)', size = 18)
ax.spines[['right', 'top']].set_visible(False)
ax.spines[['bottom', 'left']].set_linewidth(2)
ax.tick_params(width=3, labelsize = 16)
ax.legend(loc='upper right',  prop={'size': 16}, fancybox=True, framealpha=0, bbox_to_anchor = (1.1,1))

ax.set_yticks([0,0.005,0.010])
ax.set_ylim(0,0.009)
fig.tight_layout()


### Subjectwise Sigmoid

In [ ]:
from scipy.optimize import least_squares
from statsmodels.tools.tools import add_constant
import statsmodels.api as sm

def sigmoid(x, Q, k):
    return 1 / (1+ Q * np.exp(-k*x))

def residual_sigmoid(x,t,y):
    return sigmoid(t,x[0],x[1]) - y


fig, axes = plt.subplots(1,4, figsize = (10,3))
max_array = np.zeros((20,len(np.asarray(avg_time[regressor_id]))))
ax = axes[0]
for subject_id in range(len(np.asarray(avg_time[regressor_id]))):
    color_sub = cm.viridis(int(subject_id/len(np.asarray(avg_time[regressor_id]))*250))
    for regressor_index, regressor_id in enumerate(regressors_list[:20]):
        max_array[regressor_index,subject_id] = np.mean(np.asarray(avg_time[regressor_id])[:,25:175],axis=1)[subject_id]
    max_array[:,subject_id] = max_array[:,subject_id] - np.min(max_array[:,subject_id])
    max_array[:,subject_id] = max_array[:,subject_id] / np.max(max_array[:,subject_id])
    ax.plot(renyi_values,max_array[:,subject_id], color = 'grey', alpha = 0.6)
ax.plot(renyi_values,total_array, color = 'k', lw = 5)
ax.set_xscale('log')
ax.set_xlabel('Renyi Order (u.a.)', size = 15)
ax.set_ylabel('Peak of MI (bits)', size = 15)
ax.set_yticks([0,0.5,1])
ax.set_yticklabels([0,0.5,1], size = 12)
ax.spines[['right', 'top']].set_visible(0)
ax.spines[['bottom', 'left']].set_linewidth(1.3)
ax.tick_params(width=1.5, labelsize = 12)
#ax.legend()
#ax.scatter(1,reg_max[-1],s = 200,color = 'k', label = 'Shannon Entropy')
ax.set_ylim(0,1)
fig.tight_layout()

x = np.arange(1000)/100 - 5
x0 = np.asarray([0,0])
xData = np.log(renyi_values)
bounds_width = np.asarray([5, 5])
bounds = (x0 - bounds_width,x0 + bounds_width)
inflexion_point_list = []
slope_list = []


for subject_index in range(len(np.asarray(avg_time[regressor_id]))):
    color_sub = cm.viridis(int(subject_index/len(np.asarray(avg_time[regressor_id]))*250))
    yData = max_array[:,subject_index]
    res_lsq = least_squares(residual_sigmoid, x0, args=(xData, yData),
                            bounds = bounds, method='trf')
    Q,k = res_lsq.x
    inflexion_point = np.log(Q) / k
    inflexion_point_list.append(inflexion_point)
    slope_list.append(k)
    #ax.scatter(xData, yData)
    axes[1].plot(np.exp(x), sigmoid(x, Q, k), color = 'grey', alpha = 0.6)
inflexion_point_list, slope_list = np.asarray(inflexion_point_list), np.asarray(slope_list)
axes[2].hist(inflexion_point_list, color = 'grey', density = True, bins = 13)
axes[3].hist(slope_list, color = 'grey', density = True, bins = 13)

axes[1].plot(np.exp(x), timecourse_total, color = 'k', lw = 5)
axes[2].vlines(inflexion_point_total,0,2, color = 'k', lw = 5)
axes[3].vlines(slope_total,0,2, color = 'k', lw = 5)


for i in range(4):
    ax = axes[i]
    ax.spines[['right', 'top']].set_visible(0)
    ax.spines[['bottom', 'left']].set_linewidth(1.3)


ax = axes[0]
ax.set_title('normalized data')
ax.set_xlabel('Renyi values', size = 12)
ax.set_xscale('log')
ax.set_ylabel('Normalized Peak of MI (bits)', size = 12)
ax.set_yticks([0,0.5,1])
ax.set_yticklabels([0,0.5,1], size = 12)
ax.set_ylim([0,1.1])

ax = axes[1]
ax.set_title('Sigmoid model')
ax.set_xlabel('Renyi values', size = 12)
ax.set_xscale('log')
ax.set_ylabel('Normalized Peak of MI (bits)', size = 12)
ax.set_yticks([0,0.5,1])
ax.set_yticklabels([0,0.5,1], size = 12)

ax = axes[2]
ax.set_title('Inflexion point')
ax.set_xlabel('Renyi values', size = 12)
ax.set_xticks([-1,0,1,2])
ax.set_xticklabels([0.1,1,10,100], size = 12)
ax.set_ylabel('Density', size = 12)
ax.set_yticks([0,0.5,1])
ax.set_yticklabels([0,0.5,1], size = 12)

ax = axes[3]
ax.set_title('Slope')
ax.set_xlabel('Slope (a.u.)', size = 12)
ax.set_xticks([-2,0,2])
ax.set_xticklabels([0.001,0.1,10], size = 12)
ax.set_ylabel('Density', size = 12)
ax.set_yticks([0,0.5,1])
ax.set_yticklabels([0,0.5,1], size = 12)

fig.tight_layout()

## Group Subject Together

In [ ]:
meta_mi = dict()
data_dict = mi_union_thres
indices_division = []
start = 110 #110, 75
increment = 50 #50, 25
n_iteration = 1 #3
apply_baseline = False
baselines = 0, 50
time_window_list = []
for i in range(n_iteration): 
    indices = [start + increment*i,start + increment*(i+1)]
    indices_division.append([indices[0],indices[1]])
    start_time = round((timecourse_array/100)[indices[0]],3)
    end_time = round((timecourse_array/100)[indices[1]],3)
    time_window_list.append((start_time, end_time))
#indices_division = [[75,100],[100,150]]

for subject_index, subject_id in enumerate(data_dict.keys()):
    mi_subject = []
    mi_early_subject = []
    mi_late_subject = []
    mi_difference_subject = []
    mi_max_subject = []
    raw_subject = []
    loc_sub = locations_bipolar[subject_id]
    chan_sub = channels_bipolar[subject_id]
    for regressor_index, regressor in enumerate(data_dict[subject_id]):
        mi_time_subject = []
        mi_timediff_subject = []
        baseline_values = data_dict[subject_id][regressor]['data'].T[:,baselines[0]:baselines[1]].mean(1)[np.newaxis,:]
        for div in indices_division:
            mi_time_subject.append(data_dict[subject_id][regressor]['data'].T[:,div[0]:div[1]].mean(1))
            mi_timediff_subject.append((data_dict[subject_id][regressor]['data'].T[:,div[0]:div[1]] - data_dict[subject_id][regressor]['data'].T[:,div[0]:div[1]]).mean(1))
        if apply_baseline:
            mi_subject.append(np.asarray(mi_time_subject) - baseline_values)
            mi_early_subject.append((data_dict[subject_id][regressor]['data'] - baseline_values).T[:,50:75].mean(1))
            mi_late_subject.append((data_dict[subject_id][regressor]['data'] - baseline_values).T[:,125:150].mean(1))
            mi_difference_subject.append(np.asarray(mi_timediff_subject))
            raw_subject.append((data_dict[subject_id][regressor]['data'] - baseline_values).T)
            mi_max_subject.append((data_dict[subject_id][regressor]['data'] - baseline_values).max(0))
        else:
            mi_subject.append(np.asarray(mi_time_subject))
            mi_early_subject.append(data_dict[subject_id][regressor]['data'].T[:,50:75].mean(1))
            mi_late_subject.append(data_dict[subject_id][regressor]['data'].T[:,125:150].mean(1))
            mi_difference_subject.append(np.asarray(mi_timediff_subject))
            raw_subject.append(data_dict[subject_id][regressor]['data'].T)
            mi_max_subject.append(data_dict[subject_id][regressor]['data'].max(0))
    mi_early_subject = np.asarray(mi_early_subject)
    mi_late_subject = np.asarray(mi_late_subject)
    raw_subject = np.asarray(raw_subject)
    mi_max_subject = np.asarray(mi_max_subject)
    mi_subject = np.vstack(np.asarray(mi_subject).swapaxes(0,1))
    mi_difference_subject = np.vstack(np.asarray(mi_difference_subject).swapaxes(0,1))
    rois = data_dict[subject_id][regressor]['names']
    time_array = timecourse_array


    meta_roi = []
    for roi in rois:
        meta_roi.append(str(subject_id) + '-' + roi)

    meta_loc = []
    for loc_xyz, loc_name in zip(loc_sub.T,chan_sub):
        new_name = str(subject_id) + '-' + loc_name.replace('-','||')
        if new_name in meta_roi:
            meta_loc.append(loc_xyz)
    meta_loc = np.asarray(meta_loc).T

    if not 'early' in meta_mi:
        meta_mi['early'] = mi_early_subject
        meta_mi['late'] = mi_late_subject
        meta_mi['data'] = mi_subject
        meta_mi['raw'] = raw_subject
        meta_mi['max'] = mi_max_subject
        meta_mi['difference'] = mi_difference_subject
        meta_mi['names'] = meta_roi
        meta_mi['loc'] = meta_loc
        
    else:
        meta_mi['early'] = np.hstack([meta_mi['early'], mi_early_subject])
        meta_mi['late'] = np.hstack([meta_mi['late'], mi_late_subject])
        meta_mi['data'] = np.hstack([meta_mi['data'], mi_subject])
        meta_mi['raw'] = np.hstack([meta_mi['raw'], raw_subject])
        meta_mi['max'] = np.hstack([meta_mi['max'], mi_max_subject])
        meta_mi['difference'] = np.hstack([meta_mi['difference'], mi_difference_subject])
        meta_mi['names'] += meta_roi
        if len(meta_loc.shape) > 1:
            meta_mi['loc'] = np.hstack([meta_mi['loc'], meta_loc])
        else: 
            True

    meta_mi['early'] = np.asarray(meta_mi['early'])
    meta_mi['late'] = np.asarray(meta_mi['late'])
    meta_mi['data'] = np.asarray(meta_mi['data'])
    meta_mi['difference'] = np.asarray(meta_mi['difference'])

## Best Renyi per channel

In [ ]:
fig, ax = plt.subplots(figsize = (6.5,4.5))

'''
reg_color4 = np.asarray([cm.bwr(i) for i in np.arange(20)/20])
reg_color4[10] = [0.95,0.95,0.95,1]
reg_color4[11] = [0.95,0.95,0.95,1]
reg_color4[12] = [0.95,0.95,0.95,1]
'''

data_best_ref = meta_mi['data']

for i in range(data_best_ref.shape[0] - 1):
    n_occurences = np.sum((data_best_ref[:20,:].argmax(0) == i))
    if i != 11:
        #n_occurences = np.sum((meta_mi['raw'][:20,30:100,:].mean(-1).argmax(0) == i))
        ax.bar(i, n_occurences/data_best_ref.shape[1], color = reg_color[i], width = 1)
    else:
        ax.bar(i, n_occurences/data_best_ref.shape[1], color = color_shannon, width = 1)
#Nomenclature
ax.text(9,0.027,'Shannon\n Entropy', color = color_shannon, size = 16)


#ax.set_title('Distribution of Best Renyi Value', size = 18)
ax.spines[['right', 'top']].set_visible(False)
ax.spines[['bottom', 'left']].set_linewidth(2)
ax.tick_params(width=3, labelsize = 16)

#ticks
ax.set_xticks(np.arange(10) * 2)
ax.set_xticklabels(np.asarray(renyi_values)[list(np.arange(10) * 2)], rotation = 0)
ax.set_yticks([0,0.1,0.2])

ax.set_xlabel('Renyi Order (α)', size = 20)
ax.set_ylabel('Channels Density', size = 20)
ax.spines[['right', 'top']].set_visible(False)
ax.spines[['bottom', 'left']].set_linewidth(2)
fig.patch.set_facecolor((0.941, 0.969, 1.0))
ax.set_facecolor((0.941, 0.969, 1.0))

fig.tight_layout()

In [ ]:
data_best_plot = dict()

data_best_plot['name'] = []
data_best_plot['loc'] = []
data_best_plot['val'] = []

for data, channel, loc in zip(meta_mi['data'].T, meta_mi['names'], meta_mi['loc'].T):
    val = data[:-1].argmax() + 1
    data_best_plot['name'].append(channel)
    data_best_plot['loc'].append(loc)
    data_best_plot['val'].append(val)
    
data_best_plot['name'] = np.asarray(data_best_plot['name'])
data_best_plot['loc'] = np.asarray(data_best_plot['loc']).T
data_best_plot['val'] = np.asarray(data_best_plot['val'])

brainplot_file = 'brainplotting/best_renyi.pickle'
with open(brainplot_file, 'wb') as file:
    pickle.dump(data_best_plot,file)

## NMF Clustering

### Grid Search for NMF Hyperparameters

In [ ]:
from sklearn.decomposition import NMF, FactorAnalysis
from scipy.special import softmax
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score, silhouette_samples

number_components = np.arange(2,5)
alpha_W_list = np.logspace(-5,0.99,10)
alpha_H_list = np.logspace(-5,0.99,10)
alpha_l1_list = np.logspace(-5,0,10)
score_matrix = np.zeros([3,len(number_components),len(alpha_W_list),len(alpha_H_list),len(alpha_l1_list)])

V = meta_mi['max'][:len(renyi_values),:]
#V = meta_mi['data'][:len(renyi_values),:]
reyni_val_array = np.arange(V.shape[0])
V = V - V.min(axis=0)
V /= V.max(axis=0)
V = V

for i_n,i_W,i_H,i_l1 in product(np.arange(len(number_components)), np.arange(len(alpha_W_list)), np.arange(len(alpha_H_list)), np.arange(len(alpha_l1_list))):
        n_components = number_components[i_n]
        alpha_W = alpha_W_list[i_W]
        alpha_H = alpha_H_list[i_H]
        l1_ratio = alpha_l1_list[i_l1]

        nmf = NMF(n_components=n_components, init=None, solver='mu', beta_loss='frobenius', 
                tol=0.00001, max_iter=5000, random_state=0, 
                alpha_W=alpha_W, alpha_H=alpha_H, l1_ratio=l1_ratio, 
                verbose=0, shuffle=True) 


        reyni_NMF = nmf.fit_transform(V)
        unique_components = nmf.components_
        V_est =  reyni_NMF@unique_components
        pred_labels = softmax(unique_components,axis=0).argmax(0)


        if len(set(pred_labels)) > 1:
                silhouette = silhouette_score(V.T,pred_labels)
                dbs = davies_bouldin_score(V.T,pred_labels)
                chs = calinski_harabasz_score(V.T,pred_labels)
        else:
                silhouette = 0
                dbs = 100
                chs = 0

        score_matrix[:,i_n, i_W, i_H, i_l1] = np.array([silhouette,dbs,chs])


max_index_flats = [np.argmax(score_matrix[0]), np.argmin(score_matrix[1]), np.argmax(score_matrix[2])]
max_index_5ds = [np.unravel_index(max_index_flats[0], score_matrix[0].shape), np.unravel_index(max_index_flats[1], score_matrix[1].shape), np.unravel_index(max_index_flats[2], score_matrix[2].shape)]
optimal_sets = max_index_5ds

### Compute NMF

In [ ]:
V = meta_mi['max'][:len(renyi_values),:]
#V = meta_mi['data'][:len(renyi_values),:]
V = V - V.min(axis=0)
V /= V.max(axis=0)
selection_channels = np.arange(V.shape[1]) #np.random.choice(np.arange(633), 633//2) or np.arange(633)
V = V[:,selection_channels]
reyni_val_array = np.arange(V.shape[0])
raw_data = meta_mi['raw']

n_components = 3
alpha_W = 0.1
alpha_H = 0.1
l1_ratio = 0.5

set_index = 0
optimal_set = optimal_sets[set_index]
n_components = number_components[optimal_set[0]]
alpha_W = alpha_W_list[optimal_set[1]]
alpha_H = alpha_H_list[optimal_set[2]]
l1_ratio = alpha_l1_list[optimal_set[3]]

nmf = NMF(n_components=n_components, init=None, solver='mu', beta_loss='frobenius', 
        tol=0.00001, max_iter=2000, random_state=0, 
        alpha_W=alpha_W, alpha_H=alpha_H, l1_ratio=l1_ratio, 
        verbose=0, shuffle=True) #3, mu, kullback, w0.01, h0.00001, l10.001


#nmf = FactorAnalysis(n_components=n_components, tol = 0.00001)

reyni_NMF = nmf.fit_transform(V)
unique_components = nmf.components_
V_est =  reyni_NMF@unique_components
norm_unique_comps = unique_components / unique_components.sum(0)
pred_labels = norm_unique_comps.argmax(0)
nmf_labels = norm_unique_comps.argmax(0)
threshold_nmf = 0.65 #0.8,0.6, 0.65
nmf_labels[norm_unique_comps.max(0) < threshold_nmf] = n_components 
#nmf_labels = pred_labels
#nmf_labels = np.zeros(nmf_labels.shape[0], int)


silhouette = silhouette_score(V.T,pred_labels)
silhouette_per_sample = silhouette_samples(V.T, pred_labels)
calinski = calinski_harabasz_score(V.T,pred_labels)
davies = davies_bouldin_score(V.T, pred_labels)
print('silhouette score: ', silhouette)
print('calinski_harabasz_score: ', calinski)
print('davies_bouldin_score: ', davies)



print('normalized membership', norm_unique_comps.max(0).mean())
comps = []
raw_comps = []
comps_names = []
max_comps = []
for i in range(len(set(nmf_labels))):
        if np.sum(nmf_labels == i) > 0:
                print(i, np.sum(nmf_labels == i), silhouette_per_sample[nmf_labels[nmf_labels == i]].mean(0))
                comps.append(V[:,nmf_labels == i])
                raw_comps.append(raw_data[:,selection_channels][:,nmf_labels == i])
                comps_names.append(np.asarray(meta_mi['names'])[selection_channels][nmf_labels == i])
                max_comps.append(raw_data[:,selection_channels][:,nmf_labels == i].max(-1))
                #comps.append(meta_dR2_filtered.T[:,softmax(unique_components,axis=0).argmax(0) == i])
n_components = len(comps)
n_components = 2

### Main Figure

#### Timecourse

In [ ]:
%matplotlib qt
norm  = False

fig = plt.figure(figsize=(10, 11))
gs = gridspec.GridSpec(n_components,2, width_ratios=[2, 1.5])  # Custom ratios
axes = []
for i_components in range(n_components):
    axes.append([fig.add_subplot(gs[i_components,0]), fig.add_subplot(gs[i_components,1])])
axes = np.asarray(axes).T

ax = axes[0]

peak_highlight = [
    [[80,0.0047,95],[98,0.008,110],[118,0.009,130]],
    [[118,0.0057,130]]
]


max_surprise = dict()
info_time_total = dict()
for component_index in range(n_components):
    info_time_total[component_index] = []
    max_surprise[component_index] = []
    for reg in range(len(regressors_list)):
        info_time = scale(raw_comps[component_index][reg,:,:],axis=1, with_mean = norm, with_std = norm)
        info_time_total[component_index].append(info_time)
        if reg >= len(renyi_values):
            max_surprise[component_index] += list(info_time.max(1))
            ax[component_index].plot(timecourse_array,np.mean(info_time,axis=0), color = color_surprisal, label = reg_label[reg], linestyle = '--', linewidth = 3, alpha = 0)
            ax[component_index].fill_between(timecourse_array,
                                np.mean(info_time,axis=0) + np.std(info_time,axis=0)/np.sqrt(info_time.shape[0]), 
                                np.mean(info_time,axis=0) - np.std(info_time,axis=0)/np.sqrt(info_time.shape[0]), 
                                color = color_surprisal, alpha = 0)
        else:
            ax[component_index].plot(timecourse_array,np.mean(info_time,axis=0), color = cluster_reg_color[component_index][reg], label = reg_label[reg])
            ax[component_index].fill_between(timecourse_array,
                                np.mean(info_time,axis=0) + np.std(info_time,axis=0)/np.sqrt(info_time.shape[0]), 
                                np.mean(info_time,axis=0) - np.std(info_time,axis=0)/np.sqrt(info_time.shape[0]), 
                                color = cluster_reg_color[component_index][reg], alpha = 0.3)
    #Nomenclature
    ax[component_index].set_xlabel('Time (s)', size = 18)
    ax[component_index].set_ylabel('Mutual information (bits)', size = 18)
    ax[component_index].spines[['right', 'top']].set_visible(False)
    ax[component_index].spines[['bottom', 'left']].set_linewidth(2)
    ax[component_index].tick_params(width=3, labelsize = 16)
    ax[component_index].set_xlim(-0.5,1.2)
    ax[component_index].set_yticks([0,0.005,0.010])
    #Yaxis
    ax[component_index].plot([0,0],[0,0.006], color = 'k', linestyle = '--')
    ax[component_index].text(-0.25,0.0063,'word\nonset', color = 'k', size = 18)
    for percentile in [95]:
        ax[component_index].plot([np.percentile(wrd_distrib,percentile),np.percentile(wrd_distrib,percentile)],[0,0.006], color = 'k', linestyle = '--')
        ax[component_index].text(np.percentile(wrd_distrib,percentile)-0.05,0.0063,str(percentile) + '%\noffset', color = 'k', size = 18)
    ax[component_index].set_ylim(0,0.0089)

    info_time_total[component_index] = np.asarray(info_time_total[component_index])
    for highlight in peak_highlight[component_index]:
        x_coor = timecourse_array[highlight[0]] 
        y_coor = highlight[1]
        label_coor = str(round(timecourse_array[highlight[2]] *1000)) + '\n ms'
        ax[component_index].text(x_coor, y_coor, label_coor,color = ['b', 'r'][component_index], size = 18)
        

argmax_comps = dict()
for component_index in range(n_components):
    ax = axes[1][component_index]
    data = raw_comps[component_index].max(-1)
    data_mean = raw_comps[component_index][:,:,126:128].mean(1).max(1) #126:128
    data_std = raw_comps[component_index][:,:,127].std(1) / np.sqrt(raw_comps[component_index].shape[1])
    
    #component_data = reyni_NMF[:,component_index]

    for renyi_val, max_mi, std_mi,renyi_id, renyi_index in zip(renyi_values, data_mean, data_std,regressors_list, np.arange(len(renyi_values))):
        ax.scatter(renyi_val, max_mi, s= 190,color = 'white',zorder = 3, alpha = 1)
        ax.scatter(renyi_val, max_mi, s= 200,color = cluster_reg_color[component_index][renyi_index], edgecolor = 'k',zorder = 3, linewidth = 0) #1.5
        ax.errorbar(renyi_val, max_mi, yerr = std_mi, color = cluster_reg_color[component_index][renyi_index],zorder = 1, linewidth = 1.5)

    ax.set_title(['Dispersion Cluster\n\n', 'Strength Cluster\n\n'][component_index], size = 30, color = ['b', 'r'][component_index])

    ax.set_xscale('log')
    ax.set_xlabel('Renyi Order (α)', size = 18)
    ax.set_ylabel('Peak of MI (bits)', size = 18)
    ax.spines[['right', 'top']].set_visible(0)
    ax.spines[['bottom', 'left']].set_linewidth(2)
    ax.tick_params(width=3, labelsize = 16)
    ax.set_yticks([0,0.005,0.010])
    #ax.set_ylim(0,0.0089)
    #ax.set_yticks([0.002,0.004,0.006])
    #ax.text(1,0.0075,'N =' + str(raw_comps[component_index].shape[1]), size = 15, color = cluster_color[component_index])

axes[1][0].text(0.02,0.005,'   ' + str(raw_comps[0].shape[1]) + '\nchannels', size = 18, color = cluster_color[0])
axes[1][1].text(0.02,0.005,'    ' + str(raw_comps[1].shape[1]) + '\nchannels', size = 18, color = cluster_color[1])


'''
axes[1][0].set_yticks([0.004,0.006,0.008])
axes[1][0].set_ylim([0.0035,0.008])
axes[1][0].text(1,0.0075,'N =' + str(raw_comps[0].shape[1]), size = 15, color = cluster_color[0])

axes[1][1].set_yticks([0.002,0.003,0.004])
axes[1][1].set_ylim([0.0016,0.004])
axes[1][1].text(1,0.0035,'N =' + str(raw_comps[1].shape[1]), size = 15, color = cluster_color[1])
'''

fig.tight_layout()

for i in range(n_components):
    print('Subject presenting component', str(i), ':', len(set([int(name.split('-')[0]) for name in comps_names[i]])))

for i in range(n_components):
    print('Shaft presenting component', str(i), ':', set([name.split('-')[1].split('_')[0] for name in comps_names[i]]))

regions = dict()
for i in range(n_components):
    regions[i] = dict()
    for name in comps_names[i]:
        sub = int(name.split('-')[0])
        chan = name.split('-')[1].split('_')[0]
        if chan in new_atlas[sub]:
            region = new_atlas[sub][chan]
            if region in regions[i]:
                regions[i][region] += 1
            else:
                regions[i][region] = 1

for i in range(n_components):
    print('=========================================')
    print('Regions presenting component', str(i), ':')
    for region in regions[i]:
        if regions[i][region] > 2:
            print(region, regions[i][region])


#### Best Renyi per channel

In [ ]:
%matplotlib qt

for component_index in range(n_components):
    fig, ax = plt.subplots(figsize = (6.5,4.5), sharex = True, sharey = True)
    for i in range(comps[component_index].shape[0]):
        n_occurences = np.sum((comps[component_index][:20,:].argmax(0) == i))
        #n_occurences = np.sum((meta_mi['raw'][:20,30:100,:].mean(-1).argmax(0) == i))
        ax.bar(i, n_occurences/comps[component_index].shape[1], color = cluster_reg_color[component_index][i], width = 1)
    #Nomenclature
    ax.set_xlabel('Renyi Order (α)', size = 18)
    ax.set_ylabel('Channels Density', size = 18)
    ax.spines[['right', 'top']].set_visible(False)
    ax.spines[['bottom', 'left']].set_linewidth(2)

    #ticks
    ax.tick_params(width=3, labelsize = 16)
    ax.set_xticks(np.arange(10) * 2)
    ax.set_xticklabels(np.asarray(renyi_values)[list(np.arange(10) * 2)], rotation = 0)
    ax.set_yticks([0,0.2,0.4,0.6])
    ax.set_ylim([0,0.65])


    fig.tight_layout()

### Supplementary Figures

#### Membership

In [ ]:
import numpy as np
from sklearn.decomposition import NMF
from sklearn.preprocessing import normalize

def compute_fuzzy_entropy(W, normalize_rows=True):
    """
    Compute the fuzzy entropy from an NMF membership matrix.

    Parameters:
        W : ndarray (n_samples, n_clusters)
            The NMF membership matrix.
        normalize_rows : bool
            Whether to normalize each row to sum to 1 (default: True).
    
    Returns:
        entropy_per_sample : ndarray (n_samples,)
            Entropy for each data point.
        mean_entropy : float
            Mean fuzzy entropy over all samples.
        normalized_entropy : float
            Mean entropy divided by log(k), i.e., normalized to [0,1].
    """
    if normalize_rows:
        P = normalize(W, norm='l1', axis=1)
    else:
        P = W

    # Avoid log(0) by setting 0 entries to a small positive value
    P_safe = np.clip(P, 1e-12, 1.0)
    
    entropy_per_sample = -np.sum(P_safe * np.log(P_safe), axis=1)
    mean_entropy = np.mean(entropy_per_sample)
    max_entropy = np.log(P.shape[1])
    normalized_entropy = mean_entropy / max_entropy

    return entropy_per_sample, mean_entropy, normalized_entropy


In [ ]:
fig, ax = plt.subplots()

membership = unique_components / np.sum(unique_components,axis = 0)

ind = np.arange(membership.shape[1]) 
cluster_membership = np.zeros(membership.shape)
cluster_membership_ordered = np.zeros(membership.shape)
for i in range(membership.shape[0]):
    cluster_membership[i,:] = membership[i,:]  # Memberships for cluster 1
sorted_indices = np.argsort(cluster_membership[0,:])
for i in range(membership.shape[0]):
    cluster_membership_ordered[i,:] = cluster_membership[i,sorted_indices]
fuzzy_entropy = (-membership*np.log(membership)).sum(1).mean(0)
elementwise_entropy, fuzzy_entropy, normed_fuzzy_entropy = compute_fuzzy_entropy(membership.T, normalize_rows=True)

cumulated = np.zeros(cluster_membership_ordered.shape[1])
for i in range(cluster_membership_ordered.shape[0]):
    ax.bar(ind, cluster_membership_ordered[i], bottom=cumulated, alpha=0.6, color = cluster_color[i],width = 1)
    cumulated += cluster_membership_ordered[i]
ax.fill_between(np.arange(cluster_membership_ordered.shape[1]), np.zeros(cluster_membership_ordered.shape[1]),(cluster_membership_ordered.max(0) < threshold_nmf).astype(int), color = 'green', alpha = 0.5)
ax.plot([-0.5,membership.shape[1]], [0.5,0.5], '--k')
ax.set_xlabel('Channels', size = 15)
ax.set_ylabel('Membership', size = 15)
ax.spines[['bottom', 'left','right', 'top']].set_linewidth(1.3)
ax.tick_params(width=1.5, labelsize = 12)
ax.set_xlim(0,membership.shape[1])
ax.legend(loc='upper right',  prop={'size': 12}, fancybox=True, framealpha=0)
ax.set_title("Normalised Fuzzy Entropy: " + str(round(normed_fuzzy_entropy,3)), size = 18)

#### Subjectwise Sigmoid

In [ ]:
subjectwise_clustering = []
for index_component in range(n_components):
    subject_cluster = dict()
    for component_name, component_data in zip(comps_names[index_component],raw_comps[index_component].swapaxes(0,1)[:,:len(renyi_values)]):
        subject_id = int(component_name.split('-')[0])
        avg_data = component_data[:,110:160].mean(1)
        avg_data = avg_data - np.min(avg_data)
        avg_data = avg_data / np.max(avg_data)
        if subject_id in subject_cluster:
            subject_cluster[subject_id].append(avg_data)
        else:
            subject_cluster[subject_id] = [avg_data]
    for subject_id in subject_cluster:
        subject_cluster[subject_id] = np.asarray(subject_cluster[subject_id]).mean(0)
    
    subjectwise_clustering.append(subject_cluster)

In [ ]:
def sigmoid(x, Q, k):
    return 1 / (1+ Q * np.exp(-k*x))

def residual_sigmoid(x,t,y):
    return sigmoid(t,x[0],x[1]) - y

x = np.arange(1000)/100 - 5
x0 = np.asarray([0,0])
xData = np.log(renyi_values)
bounds_width = np.asarray([10, 10])
bounds = (x0 - bounds_width,x0 + bounds_width)

fig, axess = plt.subplots(2,4, figsize = (10,6))
inflexion_point_dict = dict()
slope_dict = dict()
for component_index in range(n_components):
    axes = axess[component_index]
    subject_cluster = subjectwise_clustering[component_index]
    inflexion_point_list = []
    slope_list = []

    ax = axes[0]
    for subject_id in subject_cluster:
        color_sub = cm.viridis(int(subject_id/len(np.asarray(avg_time[regressor_id]))*250))
        ax.plot(renyi_values,subject_cluster[subject_id], color = 'grey', alpha = 0.6)
    axes[0].plot(renyi_values, total_clustering[component_index], color = 'k', lw = 5)
    ax.set_xscale('log')
    ax.set_xlabel('Renyi Order (u.a.)', size = 15)
    ax.set_ylabel('Peak of MI (bits)', size = 15)
    ax.set_yticks([0,0.5,1])
    ax.set_yticklabels([0,0.5,1], size = 12)
    ax.spines[['right', 'top']].set_visible(0)
    ax.spines[['bottom', 'left']].set_linewidth(1.3)
    ax.tick_params(width=1.5, labelsize = 12)
    #ax.legend()
    #ax.scatter(1,reg_max[-1],s = 200,color = 'k', label = 'Shannon Entropy')
    ax.set_ylim(0,1)
    fig.tight_layout()


    for subject_index in subject_cluster:
        color_sub = cm.viridis(int(subject_index/len(subject_cluster)*250))
        yData = subject_cluster[subject_index]
        res_lsq = least_squares(residual_sigmoid, x0, args=(xData, yData),
                                bounds = bounds, method='trf')
        Q,k = res_lsq.x
        inflexion_point = np.log(Q) / k
        inflexion_point_list.append(inflexion_point)
        slope_list.append(k)
        if subject_index in inflexion_point_dict:
            inflexion_point_dict[subject_index].append(inflexion_point)
            slope_dict[subject_index].append(k)
        else: 
            inflexion_point_dict[subject_index] = [inflexion_point]
            slope_dict[subject_index] = [k]
        axes[1].plot(np.exp(x), sigmoid(x, Q, k), color = 'grey', alpha = 0.6)
    inflexion_point_list, slope_list = np.asarray(inflexion_point_list), np.asarray(slope_list)
    axes[2].hist(inflexion_point_list, color = 'grey', density = True, bins = 8)
    axes[3].hist(slope_list, color = 'grey', density = True, bins = 8)

    axes[1].plot(np.exp(x), timecourse_total[component_index], color = 'k', lw = 5)
    axes[2].vlines(inflexion_point_total[component_index],0,2, color = 'k', lw = 5)
    axes[3].vlines(slope_total[component_index],0,2, color = 'k', lw = 5)

    for i in range(4):
        ax = axes[i]
        ax.spines[['right', 'top']].set_visible(0)
        ax.spines[['bottom', 'left']].set_linewidth(1.3)


    ax = axes[0]
    ax.set_title('normalized data')
    ax.set_xlabel('Renyi values', size = 12)
    ax.set_xscale('log')
    ax.set_ylabel('Normalized Peak of MI (bits)', size = 12)
    ax.set_yticks([0,0.5,1])
    ax.set_yticklabels([0,0.5,1], size = 12)
    ax.set_ylim([0,1.1])

    ax = axes[1]
    ax.set_title('Sigmoid model')
    ax.set_xlabel('Renyi values', size = 12)
    ax.set_xscale('log')
    ax.set_ylabel('Normalized Peak of MI (bits)', size = 12)
    ax.set_yticks([0,0.5,1])
    ax.set_yticklabels([0,0.5,1], size = 12)

    ax = axes[2]
    ax.set_title('Inflexion point')
    ax.set_xlabel('Renyi values', size = 12)
    ax.set_xticks([-1,0,1,2])
    ax.set_xticklabels([0.1,1,10,100], size = 12)
    ax.set_ylabel('Density', size = 12)
    ax.set_yticks([0,0.5,1])
    ax.set_yticklabels([0,0.5,1], size = 12)

    ax = axes[3]
    ax.set_title('Slope')
    ax.set_xlabel('Slope (a.u.)', size = 12)
    ax.set_xticks([-2,0,2,4])
    ax.set_xticklabels([0.001,0.1,10,1000], size = 12)
    ax.set_ylabel('Density', size = 12)
    ax.set_yticks([0,0.5,1])
    ax.set_yticklabels([0,0.5,1], size = 12)
    ax.set_xlim([-3.5,4.5])

fig.tight_layout()

### Save cluster

In [ ]:
clustering_type = 'NMF6'

for subject_index, subject_id in enumerate(mi_group):
    cluster_dict = dict()
    for i_cluster in range(len(comps_names)):
        cluster_dict[i_cluster] = []
        channels_cluster = comps_names[i_cluster]
        for channel in channels_cluster:
            sub = channel.split('-')[0]
            name = channel.split('-')[1]
            if sub == str(subject_id):
                cluster_dict[i_cluster].append(name)
    regressors_str = '_'.join(np.asarray(regressors_list).astype('str'))
    filename = 'MI_cluster/' + clustering_type + '_'  + montage + '_' +  str(subject_id) + '_' + regressors_str + '.pickle'
    with open(filename, 'wb') as file:
        pickle.dump(cluster_dict,file)


In [ ]:
for subject_index, subject_id in enumerate(mi_group):
    cluster_dict = dict()
    for i_cluster in range(len(comps_names)):
        cluster_dict[i_cluster] = {'names':[], 'data':dict()}
        for regressor in regressors_list:
            cluster_dict[i_cluster]['data'][regressor] = []
        channels_cluster = comps_names[i_cluster]
        data_cluster = raw_comps[i_cluster]
        for channel, data in zip(channels_cluster, data_cluster.swapaxes(0,1)):
            sub = channel.split('-')[0]
            name = channel.split('-')[1]
            if sub == str(subject_id):
                cluster_dict[i_cluster]['names'].append(name)
                for reg_index, regressor in enumerate(regressors_list):
                    cluster_dict[i_cluster]['data'][regressor].append(data[reg_index])
        for regressor in regressors_list:
            cluster_dict[i_cluster]['data'][regressor] = np.asarray(cluster_dict[i_cluster]['data'][regressor])
    regressors_str = '_'.join(np.asarray(regressors_list).astype('str'))
    filename = 'MI_cluster/' + clustering_type + '_fulldata_' + montage + '_' + str(subject_id) + '_' + regressors_str + '.pickle'
    with open(filename, 'wb') as file:
        pickle.dump(cluster_dict,file)

## Quartiles Analysis

### Setup

#### Load Partition

In [ ]:
# Renyi Array
offset_wrd = False

path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/renyi_wrdonly_array.pickle"
path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/accurate_reg.pickle"
path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/inaccurate_reg.pickle"
path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/confident_reg.pickle"
#path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/inconfident_reg.pickle"
#path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/certain_reg.pickle"
#path_renyi = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/uncertain_reg.pickle"
path_renyi_Q1 = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/Q1_reg.pickle"
path_renyi_Q2 = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/Q2_reg.pickle"
path_renyi_Q3 = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/Q3_reg.pickle"
path_renyi_Q4 = r"C:/Users/D-CAP/Documents/GitHub/witching-star/semantic_regressor/Q4_reg.pickle"


partition_paths = [path_renyi_Q1, path_renyi_Q2, path_renyi_Q3, path_renyi_Q4]
partition_regs = []
for partition_path in partition_paths:
    renyi_data = pickle.load(open(partition_path, 'rb'))
    data_fs = renyi_data['fs']
    X = np.roll(renyi_data['X'],4, axis=0)
    onsets = np.where(X[:,0] >0)[0]
    offsets = np.where(X[:,1] >0)[0]
    reg_renyi = np.zeros([int(X.shape[0]/data_fs*hfs), X.shape[1]])

    if offset_wrd:
        timelock_choice = offsets
    else:
        timelock_choice = onsets
    for onset in timelock_choice:
        new_onset = int(onset/data_fs*hfs)
        for renyi_index in range(X.shape[1]):
            reg_renyi[new_onset, renyi_index] = X[onset,renyi_index]

    partition_regs.append(reg_renyi)

#### Compute Epochs

In [ ]:
from spyeeg.models.ERP import ERP_class

channel_selection = []
exclude = True
filt_toggle = True
filt = [0.1,49]
components_names = comps_names[:2] 
norm_baseline = False

cluster_raw_eeg = dict()
cluster_epochs = dict()
cluster_locations = dict()
cluster_names = dict()

for partition_index in range(len(partition_regs)):
    cluster_raw_eeg[partition_index] = dict()
    cluster_epochs[partition_index] = dict()
    cluster_locations[partition_index] = dict()
    cluster_names[partition_index] = dict()

for partition_index in range(len(partition_regs)):
    for cluster in range(len(components_names)):
        cluster_raw_eeg[partition_index][cluster] = []
        cluster_epochs[partition_index][cluster] = []
        cluster_locations[partition_index][cluster] = []
        cluster_names[partition_index][cluster] = []

for subject_index, subject_id in enumerate(data_subject):
    cluster_dict = dict()
    for i_cluster in range(len(components_names)):
        cluster_dict[i_cluster] = []
        channels_cluster = components_names[i_cluster]
        for channel in channels_cluster:
            sub = channel.split('-')[0]
            name = channel.split('-')[1]
            if sub == str(subject_id):
                cluster_dict[i_cluster].append(name)
    eeg = data_bipolar[subject_id]
    eeg_Hfa = data_Hfabipolar[subject_id]
    channels = channels_bipolar[subject_id]
    locations = locations_bipolar[subject_id]

    eeg_mono = data_subject[subject_id]
    channels_mono = channels_subject[subject_id]
    locations_mono = locations_subject[subject_id]
    
    eeg_HT, channels_HT, locations_HT = select_channels(eeg,channels, locations, channel_select = channel_selection, exclude = exclude)
    eeg_HfaHT, channels_HfaHT, locations_HfaHT = select_channels(eeg_Hfa,channels, locations, channel_select = channel_selection, exclude = exclude)
    eeg_mono_HT, channels_mono_HT, locations_mono_HT = select_channels(eeg_mono,channels_mono, locations_mono, channel_select = channel_selection, exclude = exclude)
    y1 = eeg_HT[:,:]
    y1_mono = eeg_mono_HT[:,:]
    y1_Hfa = eeg_HfaHT[:,:]

    tmin = -1.0 #-1
    tmax = 2 #1.5
    step = 1 #1
    env_ref = 1 #1
    baseline_limits = [0,50]
    time_array = np.linspace(tmin, tmax,int((tmax-tmin) * hfs))

    if montage == 'mono':
        channels_name = channels_mono_HT
        y = eeg_mono_HT[:,:]
        xyz = locations_mono_HT
    elif montage == 'bipo':
        channels_name = []
        for chan_name in channels_HT:
            channels_name.append(chan_name.replace('-', '||'))
            y = eeg_HT[:,:]
            xyz = locations_HT
    elif montage == 'ica':                
        y_ica, channels_ica = ica_shaft(y1_mono, channels_mono_HT, random_state = 0)
        channels_name = channels_ica
        y = y_ica
        xyz = np.zeros(locations_mono_HT.shape)
    
    elif montage == 'Hfa_bipo':
        channels_name = []
        for chan_name in channels_HfaHT:
            channels_name.append(chan_name.replace('-', '||'))
            y = eeg_HfaHT[:,:]
            xyz = locations_HfaHT

    if filt_toggle:
        y = mne.filter.filter_data(y.T, hfs ,filt[0], filt[1], verbose = False).T

    for partition_index,partition_reg in enumerate(partition_regs):
        x = partition_reg[:,0]
        #y = y[:length,:]
        erp = ERP_class(tmin = tmin, tmax = tmax, srate=hfs)
        erp.add_events(y, x, weight_events = False, record_weight = False)
        epoched_data = np.asarray(erp.evoked).transpose(0,2,1)
        
        baseline = np.repeat(epoched_data[:,:,baseline_limits].mean(-1), epoched_data.shape[-1]).reshape(epoched_data.shape)
        if norm_baseline:
            epoched_data = epoched_data - baseline

        #baseline = epoched_data[:,:,baseline_limits]
        #baseline_avg, baseline_std = np.mean(baseline,axis = 2), np.std(baseline,axis = 2)
        #epoched_data = (epoched_data - baseline_avg[:, :,np.newaxis]) / baseline_std[:, :,np.newaxis]
        
        for cluster in cluster_dict:
            mask_cluster = [select in cluster_dict[cluster] for select in channels_name]
            y_cluster = y[:,mask_cluster]
            xyz_cluster = xyz[:,mask_cluster]
            channels_name_cluster = np.asarray(channels_name)[mask_cluster]
            epoched_cluster = epoched_data[:,mask_cluster,:]
            cluster_raw_eeg[partition_index][cluster].append(y_cluster.T)
            cluster_epochs[partition_index][cluster].append(epoched_cluster.transpose(1,0,2))
            cluster_locations[partition_index][cluster].append(xyz_cluster.T)
            cluster_names[partition_index][cluster].append(channels_name_cluster)


### Plot Quartiles

In [ ]:
%matplotlib qt

cluster_evk = dict()
cluster_normevk = dict()
n_partition = len(cluster_raw_eeg)
n_cluster = len(cluster_raw_eeg[0])
synergy_window_list = [np.arange(124,134), np.arange(128,138)] # [np.arange(125,135), np.arange(145,155)]
p_thres = 0.01
start_baseline = 0 #50
end_baseline = 50 #80
start_test = 50
end_test = 200
baseline_samples = list(np.arange(timecourse_array.shape[0]))[start_baseline:end_baseline]

cluster_index_plot = 1

clusters_plot = [cluster_index_plot]

partition_names = [['Concentrated Prediction\nUnsurprising', 'Dispersed Prediction\nSurprising', 'Dispersed Prediction\nUnsurprising', 'Concentrated Prediction\nSurprising'],
['Strong Prediction\nUnsurprising', 'Weak Prediction\nSurprising', 'Weak Prediction\nUnsurprising', 'Strong Prediction\nSurprising']][cluster_index_plot]


color_quartiles = [['turquoise', 'darkblue', 'mediumaquamarine', 'teal'],
                   ['gold', 'darkred', 'orange', 'salmon']][cluster_index_plot]

for partition in cluster_epochs:
    cluster_evk[partition] = dict()

for partition in cluster_epochs:
    for cluster in cluster_epochs[partition]:
        cluster_evk[partition][cluster] = []

for partition in cluster_epochs:
    for cluster in cluster_epochs[partition]:
        for subject_index in range(len(cluster_epochs[partition][cluster])):
            data = cluster_epochs[partition][cluster][subject_index].mean(1)
            if data.shape[0] > 0:
                cluster_evk[partition][cluster] += (list(data))
        cluster_evk[partition][cluster] = np.asarray(cluster_evk[partition][cluster])


fig, axes = plt.subplots(2,2, figsize = (7,6), sharey = True, sharex = True)
ymin_all = 0
ymax_all = 0
for partition in cluster_evk:
    cluster_normevk[partition] = dict()
    ax_index = [[1,0],[0,1],[1,1],[0,0]][partition]
    ax = axes[ax_index[0],ax_index[1]]
    for cluster in clusters_plot:
        data = cluster_evk[partition][cluster]**2
        baseline = data[:,baseline_samples]
        baseline_avg, baseline_std = np.mean(baseline,axis = 1), np.std(baseline,axis = 1)
        baseline_std = np.clip(baseline_std,1e-3,np.inf)
        #baseline_avg, baseline_std = np.mean(baseline), np.std(baseline)
        normalized_data = (data - baseline_avg[:, np.newaxis]) / baseline_std[:, np.newaxis]
        #normalized_data = (data - baseline_avg) / baseline_std
        normalized_data = data  - baseline_avg[:,np.newaxis]
        statistics1 = []
        for i in range(start_test,end_test):
            pval = stats.ttest_rel(data[:,i], baseline.mean(1), alternative = 'greater')[1]
            statistics1.append(pval)
        statistics1 = list(np.ones(end_baseline)) + list(multipletests(statistics1, method = 'fdr_tsbh')[1]) +  list(np.ones(data.shape[1] - end_test))
        statistics = np.asarray(statistics1) < p_thres
        statistics = filter_binary(statistics, min_cluster_size= 10)
        non_statistics = np.asarray(statistics1) > p_thres
        data = normalized_data
        cluster_normevk[partition][cluster] = data
        ax.plot(time_array,data.mean(0), 'grey', alpha = 0.3,linewidth = 2)
        ax.fill_between(time_array, 
                        data.mean(0) + data.std(0)/np.sqrt(data.shape[0]),
                        data.mean(0) - data.std(0)/np.sqrt(data.shape[0]),
                        alpha = 0.3, color = 'grey')
        ax.plot(time_array[statistics],data.mean(0)[statistics], color = color_quartiles[partition], linewidth = 2, label = 'cluster ' + str(cluster) + ', size = ' + str(data.shape[0]))
        ax.fill_between(time_array[statistics], 
                        data.mean(0)[statistics] + data.std(0)[statistics]/np.sqrt(data.shape[0]),
                        data.mean(0)[statistics] - data.std(0)[statistics]/np.sqrt(data.shape[0]),
                        alpha = 0.5, color = color_quartiles[partition])
        #ax.fill_between(time_array, 
        #                data.mean(0) + data.std(0)/np.sqrt(data.shape[0]),
        #                data.mean(0) - data.std(0)/np.sqrt(data.shape[0]),
        #                alpha = 0.5, color = color_quartiles[partition])
        ymin, ymax = ax.get_ylim()
        ymin_all, ymax_all = min(ymin, ymin_all), max(ymax, ymax_all)
        #Nomenclature
        ax.set_title(partition_names[partition], size = 16)

        #Xaxis
    ax.axvline(0,color = 'k', lw = 2, ls = '--')
    #ax.axvline(np.percentile(wrd_distrib,95),color = 'k', lw = 2, ls = '--')
    ax.spines[['right', 'top']].set_visible(0)
    ax.spines[['bottom', 'left']].set_linewidth(2)
    ax.tick_params(width=3, labelsize = 16)
    ax.set_xlabel('Time (s)', size = 16)
    ax.set_xlim(0.5,1.3)
    ax.set_xticks([-0.5,0,0.5,1])
    #ax.set_ylim([-0.01,0.055])

for partition in cluster_evk:
    ax_index = [[1,0],[0,1],[1,1],[0,0]][partition]
    ax = axes[ax_index[0],ax_index[1]]
    ax.fill_between(time_array[list(synergy_window_list[cluster])],ymin_all-1,ymax_all+1, zorder = -10, color = ['b','r'][cluster], alpha = 0.15, lw = 0.05, linestyle = '--')
    ax.set_ylim(ymin_all,ymax_all)
    
    
axes[1,0].set_ylabel('                                        Bipolar ERP power (μV²)', size = 16)
fig.tight_layout()


In [ ]:

for cluster in [cluster_index_plot]:
    synergy_window = synergy_window_list[cluster]
    pval = []
    for comb in [[0,2],[0,1],[0,3],[2,1],[3,1]]:
        pval.append(stats.ttest_rel((cluster_normevk[comb[0]][cluster])[:,synergy_window].mean(1), (cluster_normevk[comb[1]][cluster])[:,synergy_window].mean(1))[1]) #125:140
        #pval.append(stats.ttest_rel((cluster_normevk[comb[0]][cluster])[:,synergy_window].max(1), (cluster_normevk[comb[1]][cluster])[:,synergy_window].max(1))[1]) #125:140
    print(multipletests(pval, method = 'fdr_bh'))
    print(pval)

In [ ]:
import pingouin as pg
add1 = (cluster_normevk[2][cluster] - cluster_normevk[0][cluster])[:,synergy_window].mean(1)
add2 = (cluster_normevk[3][cluster] - cluster_normevk[0][cluster])[:,synergy_window].mean(1)
combination = (cluster_normevk[1][cluster] - cluster_normevk[0][cluster])[:,synergy_window].mean(1)

fig, axes = plt.subplots(1,2, figsize = (7,3))


ax = axes[0]
additive_effect = add1 + add2
supra_additive = combination - additive_effect
df_lin = pg.linear_regression(additive_effect, combination, add_intercept=True, weights=None, coef_only=False, alpha=0.05, as_dataframe=True, remove_na=False, relimp=True)
coefs = df_lin['coef'].values
coefs = [0,1]
Xreg = additive_effect
Yreg = coefs[0] + coefs[1] * Xreg 
Y_pred = np.polyval(coefs, Xreg)
residuals = np.abs(combination - Yreg)
ax.scatter(additive_effect, combination, c=residuals, cmap=['Blues','Reds'][cluster_index_plot], s=80, edgecolor='black')
#ax.plot([Xreg.min(), Xreg.max()], 
#        [coefs[0] + coefs[1] * Xreg.min(), coefs[0] + coefs[1] * Xreg.max()], color = 'k')
ax.plot([Xreg.min(), Xreg.max()], 
        [coefs[0] + coefs[1] * Xreg.min(), coefs[0] + coefs[1] * Xreg.max()], color = [0,0.5,0], lw = 3, ls = '--')

ax.spines[['right', 'top']].set_visible(0)
ax.spines[['bottom', 'left']].set_linewidth(2)
ax.tick_params(width=3, labelsize = 14)
ax.set_xlabel('Additive\neffect (μV²)', size = 16)
ax.set_ylabel('Combination\neffect (μV²)', size = 16)


ax = axes[1]
print(pg.ttest(combination, additive_effect,  paired = True))
ax.bar([0],height = np.mean(add1) + np.mean(add2), color = [0,0.5,0])
ax.bar([1],height = np.mean(combination), color = ['b', 'r'][cluster_index_plot])
ax.errorbar([0],np.mean(add1 + add2), yerr = np.std(add1 + add2)/np.sqrt(len(add1)), color = 'k', lw = 2)
ax.errorbar([1],np.mean(combination), yerr = np.std(combination)/np.sqrt(len(combination)), color = 'k', lw = 2)
ax.plot([0,1],[0.032,0.032], c='k', lw = 1.5)
ax.text(0.45,0.032,'*', size = 15)


ax.spines[['right', 'top']].set_visible(0)
ax.spines[['bottom', 'left']].set_linewidth(2)
ax.tick_params(width=3, labelsize = 16)
ax.set_ylabel('Power change (μV²)', size = 1)
ax.set_xticks([0,1])
ax.set_xticklabels(['Additive\neffect', 'Combination\neffect'])
ax.set_ylim(0,None)


fig.tight_layout()